In [0]:
base_path = "/Volumes/workspace/apex_retail/raw_landing_zone"

datasets = ["customer", "product", "sales"]
loads = ["historical", "incremental"]

landing_dfs = {}

for ds in datasets:
    for load in loads:
        raw_path = f"{base_path}/raw/{ds}/{load}/"
        df = spark.read.csv(raw_path, header=True, inferSchema=False)
        landing_dfs[f"{ds}_{load}"] = df

        landing_path = f"{base_path}/landing/{ds}/{load}/"
        df.write.mode("overwrite").parquet(landing_path)

        print(f"{ds}_{load}: {df.count()} rows -> written as Parquet to {landing_path}")

customer_historical: 1052 rows -> written as Parquet to /Volumes/workspace/apex_retail/raw_landing_zone/landing/customer/historical/
customer_incremental: 1053 rows -> written as Parquet to /Volumes/workspace/apex_retail/raw_landing_zone/landing/customer/incremental/
product_historical: 1043 rows -> written as Parquet to /Volumes/workspace/apex_retail/raw_landing_zone/landing/product/historical/
product_incremental: 1041 rows -> written as Parquet to /Volumes/workspace/apex_retail/raw_landing_zone/landing/product/incremental/
sales_historical: 1002 rows -> written as Parquet to /Volumes/workspace/apex_retail/raw_landing_zone/landing/sales/historical/
sales_incremental: 1000 rows -> written as Parquet to /Volumes/workspace/apex_retail/raw_landing_zone/landing/sales/incremental/


In [0]:
# Explicit mapping: (dataset, load) -> actual audit_landing filename
# (matches your real filenames, including the inconsistent naming)
audit_landing_files = {
    ("customer", "historical"):   "customer_historical_audit.csv",
    ("customer", "incremental"):  "customer_incrementalaudit.csv",
    ("product",  "historical"):   "product_historical_audit.csv",
    ("product",  "incremental"):  "product_incrementalaudit.csv",
    ("sales",    "historical"):   "sales_historical_audit.csv",
    ("sales",    "incremental"):  "sales_incrementalaudit.csv",
}

audit_report = []

for ds in datasets:
    for load in loads:
        actual_count = landing_dfs[f"{ds}_{load}"].count()

        audit_file = audit_landing_files[(ds, load)]
        audit_df = spark.read.csv(
            f"{base_path}/audit_landing/{audit_file}",
            header=True, inferSchema=False
        )
        expected_count = int(audit_df.collect()[0]["row_count"])

        status = "PASS" if actual_count == expected_count else "FAIL"
        audit_report.append({
            "table_name": f"{ds}_{load}",
            "expected_row_count": expected_count,
            "actual_row_count": actual_count,
            "status": status
        })

report_df = spark.createDataFrame(audit_report)
display(report_df)

failures = [r for r in audit_report if r["status"] == "FAIL"]
if failures:
    raise Exception(f"Landing audit FAILED for: {[f['table_name'] for f in failures]}")
else:
    print("✅ All datasets passed Landing audit validation.")

actual_row_count,expected_row_count,status,table_name
1052,1052,PASS,customer_historical
1053,1053,PASS,customer_incremental
1043,1043,PASS,product_historical
1041,1041,PASS,product_incremental
1002,1002,PASS,sales_historical
1000,1000,PASS,sales_incremental


✅ All datasets passed Landing audit validation.
